# Earthquake Classifier — 1D CNN on Raw Waveforms
Same binary task as the 2D CNN but operates on the raw 6000-sample seismic trace
rather than a compressed spectrogram. Uses Conv1d layers with progressively
larger kernels to capture wave patterns at 100 Hz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay,
)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
EPOCHS     = 30
LR         = 1e-3
PATIENCE   = 5
CKPT_PATH  = "best_model_1d.pt"

print(f"Using device: {DEVICE}")

## Load & normalise data

In [ ]:
X    = np.load("../data/waveforms.npy")                    # (N, 6000), already z-scored per trace
meta = pd.read_csv("../data/metadata.csv", low_memory=False)

# Add channel dim → (N, 1, 6000) as required by Conv1d
X = X[:, np.newaxis, :]                                    # (N, 1, 6000)

# Global normalisation across the training set
train_mask = (meta["split"] == "train").values
mu  = float(X[train_mask].mean())
std = float(X[train_mask].std())
X   = ((X - mu) / (std + 1e-8)).astype(np.float32)

labels = meta["label"].values.astype(np.float32)

print(f"Waveforms    : {X.shape}  dtype={X.dtype}")
print(f"Earthquakes  : {(labels==1).sum():,}  |  Noise: {(labels==0).sum():,}")
print(f"Train / Val / Test : {train_mask.sum():,} / {(meta['split']=='val').sum():,} / {(meta['split']=='test').sum():,}")

## Dataset & DataLoaders

In [ ]:
class WaveformDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):          return len(self.y)
    def __getitem__(self, i):   return self.X[i], self.y[i]


def make_loader(split_name, shuffle=False):
    mask = (meta["split"] == split_name).values
    return DataLoader(
        WaveformDataset(X[mask], labels[mask]),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

train_loader = make_loader("train", shuffle=True)
val_loader   = make_loader("val")
test_loader  = make_loader("test")
print(f"Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

## Model — 1D CNN
Input: (batch, 1, 6000). Four conv blocks with aggressive pooling reduce the
sequence from 6000 → 1500 → 375 → 75 → 4 before the classifier head.
Kernel sizes decrease as the receptive field grows with depth.

In [ ]:
class EarthquakeCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: (1, 6000) → (32, 1500)
            nn.Conv1d(1,   32,  kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),
            # Block 2: (32, 1500) → (64, 375)
            nn.Conv1d(32,  64,  kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),
            # Block 3: (64, 375) → (128, 75)
            nn.Conv1d(64,  128, kernel_size=7,  padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(5),
            # Block 4: (128, 75) → (256, 4)
            nn.Conv1d(128, 256, kernel_size=5,  padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),           # 256 * 4 = 1024
            nn.Linear(1024, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


model = EarthquakeCNN1D().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5
)

history          = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1      = 0.0
patience_counter = 0

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(y_batch)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_probs, val_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item() * len(y_batch)
                val_probs.append(torch.sigmoid(logits).cpu().numpy())
                val_true.append(y_batch.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_probs = np.concatenate(val_probs)
        val_true  = np.concatenate(val_true)
        val_f1    = f1_score(val_true, (val_probs >= 0.5).astype(int))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:2d}/{EPOCHS}  "
              f"train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  "
              f"val_F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), CKPT_PATH)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

except KeyboardInterrupt:
    print(f"\nStopped manually at epoch {epoch}.")

print(f"Best val F1: {best_val_f1:.4f}  —  checkpoint saved to {CKPT_PATH}")

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["val_f1"])
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")
axes[1].set_title("Validation F1")
plt.tight_layout(); plt.show()

## Test set evaluation

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

test_probs, test_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(DEVICE))
        test_probs.append(torch.sigmoid(logits).cpu().numpy())
        test_true.append(y_batch.numpy())

test_probs = np.concatenate(test_probs)
test_true  = np.concatenate(test_true)
test_preds = (test_probs >= 0.5).astype(int)

## Metrics

In [ ]:
print("=" * 50)
print("Test set metrics  —  1D CNN on raw waveforms")
print("=" * 50)
print(classification_report(test_true, test_preds, target_names=["Noise", "Earthquake"], digits=4))
print(f"F1 (macro)   : {f1_score(test_true, test_preds, average='macro'):.4f}")
print(f"F1 (weighted): {f1_score(test_true, test_preds, average='weighted'):.4f}")
print(f"Accuracy     : {accuracy_score(test_true, test_preds):.4f}")
print(f"Precision    : {precision_score(test_true, test_preds):.4f}")
print(f"Recall       : {recall_score(test_true, test_preds):.4f}")
print(f"AUC-ROC      : {roc_auc_score(test_true, test_probs):.4f}")

## Confusion matrix & ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(
    confusion_matrix(test_true, test_preds),
    display_labels=["Noise", "Earthquake"],
).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix — Test Set (1D CNN)")
RocCurveDisplay.from_predictions(test_true, test_probs, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].set_title("ROC Curve — Test Set (1D CNN)")
axes[1].legend()
plt.tight_layout(); plt.show()